# Architecture-dependent physical costs

FlagQuantum composes hardware assumptions, data layouts, magic-state factories
and code-error models to estimate a declared logical workload. Logical budgets
come from the existing resource interface. This notebook studies physical scalar
models. Circuit construction continues through `circuit.append(op)`.

Logical costs include algorithm qubits and logical gate categories. Physical
costs include physical qubits, wall-clock duration and first-order failure
probability. Primitive counts and logical costs are recorded independently of
these physical assumptions.


## `PhysicalCostModel`

We contain all the components for doing an estimation of physical costs within the
`flagquantum.qec.resources.physical_cost.PhysicalCostModel` class.
The model is parameterized by 1) properties of the target hardware architecture and 2) Execution protocol design choices.
We further factor the design choices into a) the data block design for storing
algorithm qubits, b) the magic state factory construction, and c) the error suppression
ability of the code. In total, the model is composed of

 - `physical_params: PhysicalParameters`
 - `data_block: SurfaceCodeDataLayout`
 - `factory: MagicStateFactory`
 - `qec_scheme: SurfaceCodeScheme`

Each method for computing physical costs take logical workload inputs: the number of
algorithm qubits and the number of algorithm gates. Output quantities
include the wall-clock time, the number of physical qubits, and the probability of failure
due to the physical realization of the algorithm.

In [ ]:
from flagquantum.compiler._component_trace.resource_model import (
    GateCost,
    ModeledResources,
)

# Declare logical costs independently of physical architecture assumptions.
alg = ModeledResources(peak_logical_qubits=100, gates=GateCost(toffoli=10**8))


In [ ]:
from flagquantum.qec.resources.physical_cost import PhysicalCostModel

cost_model = PhysicalCostModel.gidney_fowler(data_distance=25)
print(
    f"Duration {cost_model.duration_hours(alg):.2f} hr",
    f"\nPhysical Qubits {cost_model.n_physical_qubits(alg):,d}",
    f"\nError {cost_model.error(alg):.2g}",
)


In [ ]:
cost_model_2 = PhysicalCostModel.beverland(data_distance=25)
print(
    f"Duration {cost_model_2.duration_hours(alg):.2f} hr",
    f"\nPhysical Qubits {cost_model_2.n_physical_qubits(alg):,d}",
    f"\nError {cost_model_2.error(alg):.2g}",
)


## `SurfaceCodeDataLayout`

The `SurfaceCodeDataLayout` interface prescribes methods for modeling the costs of the data block of a surface code compilation.

The number of algorithm qubits is reported by FlagQuantum as a logical cost of a bloq. The
surface code is a rate-1 code, so each bit of data needs at least one surface code tile. Due
to locality constraints imposed by the 2D surface code combined with the need to interact
qubits that aren’t necessarily local, additional tiles are needed to actually execute a program.

Each data block is responsible for reporting the number of tiles required to store a certain
number of algorithm qubits; as well as the number of time steps required to consume a magic
state. Different data blocks exist in the literature, and data block provides a different
space-time tradeoff.

The space occupied by the data block is to be contrasted with the space used for magic
state distillation.

In [ ]:
from flagquantum.qec.resources.data_layout import RoutingDataLayout, SquareDataLayout

simple_data_block = RoutingDataLayout(distance=17, routing_overhead=0.5)
simple_data_block.physical_qubits(logical_qubits=100)


In [ ]:
fast_data_block = SquareDataLayout(distance=17)
fast_data_block.physical_qubits(logical_qubits=100)


## `MagicStateFactory`

The `MagicStateFactory` interface prescribes methods for modeling the costs of the magic state factories of a surface code compilation.

An important consideration for a surface code compilation is how to execute arbitrary gates
to run the desired algorithm. The surface code can execute Clifford gates in a fault-tolerant
manner. Non-Clifford gates like the T gate, Toffoli or CCZ gate, or non-Clifford rotation
gates require more expensive gadgets to implement. Executing a T or CCZ gate requires first
using the technique of state distillation in an area of the computation called a "magic state
factory" to distill a noisy T or CCZ state into a "magic state" of sufficiently low error.
Such quantum states can be used to enact the non-Clifford quantum gate through gate
teleportation.

Magic state production is thought to be an important runtime and qubit-count bottleneck in
foreseeable fault-tolerant quantum computers.

This abstract interface specifies that each magic state factory must report its required
number of physical qubits, the number of error correction cycles to produce enough magic
states to enact a given number of logical gates and an error model, and the expected error
associated with generating those magic states.

In [ ]:
from flagquantum.qec.resources.error_model import LogicalErrorModel, SurfaceCodeScheme

err_model = LogicalErrorModel(
    qec_scheme=SurfaceCodeScheme.gidney_fowler(), physical_error=1e-3
)


In [ ]:
from flagquantum.qec.resources.ccz_to_t_factory import CCZToTFactory

gate_counts = GateCost(toffoli=10**9)
ccz_factory = CCZToTFactory(level_one_distance=15, level_two_distance=31)
ccz_factory.n_cycles(gate_counts, logical_error_model=err_model)


## `SurfaceCodeScheme`

Error correcting codes suppress error but do not eliminate it. The construction of the
surface code takes a parameter $d$ that scales the amount of error suppression by using more
physical qubits per tile. The exact relationship between $d$ and the resulting logical
error rate will likely be experimentally determined when sufficiently large quantum
computers are available, but in the interim we can model the relationship with an
equation of the form
\begin{equation} \label{eq:logical_error_rate}
p_l(d) = A \left(\frac{p_p}{p^*}\right)^\frac{d + 1}{2}
\end{equation}
relating the logical error rate $p_l$ to the physical error rate $p_p$
for a code distance $d$. The coefficients $A$ and $p^*$ can be fit to
numerical simulations of a surface code, but common resource estimates in the
literature use back of the envelope values. $p^*$ is sometimes identified with
the error threshold of the surface code. Sometimes $p^*/p_p = \Lambda$ is treated as the fit
parameter.

In FlagQuantum, we provide presets for Fowler parameters $A=0.1$, $p^*=0.01$ and
Beverland parameters $A=0.03$, $p^*=0.01$.


In [ ]:
print("Gidney Fowler    ", SurfaceCodeScheme.gidney_fowler())
print("Beverland et al. ", SurfaceCodeScheme.beverland())


## `PhysicalParameters`

The physical parameters of the device include the physical error rate and the cycle time.

In [ ]:
from flagquantum.qec.resources.physical_parameters import PhysicalParameters

print("Beverland et al.", PhysicalParameters.beverland())
print("Gidney Fowler   ", PhysicalParameters.gidney_fowler())


## Modeling details

### Time costs

Execution cycles equal the larger of factory production cycles and data-layout
consumption cycles. The physical parameters set the duration of each cycle.
The literature model assumes magic-state generation is slower than the reaction
limit.


### Space costs

The number of physical qubits is the sum of the number of factory qubits and data block qubits.

### Error

We assume the constituent error probabilities are sufficiently low to permit a first-order
approximation for combining sources of error. The total error is the sum of error probabilities
due to magic state production (via `factory`) and data errors (via `data_block`). Note that
the total error in data storage depends on the number of cycles, which depends on the
factory design.

## Mixing and Matching

In this section of the notebook, we show how the various components can be interchanged to explore the variety of the modelling space -- even when just using the default parameters and a fixed code distance $d$. Additional variety can be found by adjusting the design parameters.

In [ ]:
from flagquantum.qec.resources.data_layout import (
    SingleHallwayDataLayout,
    SurfaceCodeDataLayout,
)
from flagquantum.qec.resources.fifteen_to_one import FifteenToOneFactory
from flagquantum.qec.resources.magic_state_factory import MagicStateFactory

data_distance = 25
data_blocks: list[SurfaceCodeDataLayout] = [
    RoutingDataLayout(distance=data_distance),
    SingleHallwayDataLayout(distance=data_distance),
    SquareDataLayout(distance=data_distance),
]
factories: list[MagicStateFactory] = [
    CCZToTFactory(),
    FifteenToOneFactory(data_distance, data_distance, data_distance),
]
schemes = [SurfaceCodeScheme.gidney_fowler(), SurfaceCodeScheme.beverland()]
phys_params = [
    PhysicalParameters.beverland(),
    PhysicalParameters.beverland(optimistic_error_rate=True),
    PhysicalParameters.gidney_fowler(),
    PhysicalParameters.gidney_fowler(optimistic_error_rate=True),
]


In [ ]:
import itertools

import numpy as np
from matplotlib import pyplot as plt

duration_hr_estimates: list[float] = []
n_physical_qubits_estimates: list[int] = []
for data_block, factory, scheme, phys in itertools.product(
    data_blocks, factories, schemes, phys_params
):
    model = PhysicalCostModel(
        data_block=data_block, factory=factory, qec_scheme=scheme, physical_params=phys
    )
    duration_hr_estimates.append(model.duration_hours(alg))
    n_physical_qubits_estimates.append(model.n_physical_qubits(alg))

plt.scatter(np.array(n_physical_qubits_estimates) / 1e6, duration_hr_estimates)
plt.xlabel("Physical qubits / million")
plt.ylabel("Duration / hr")
plt.tight_layout()


## Source and numerical evidence

Adapted from the complete 21-cell physical-cost notebook at Qualtran v0.7.0,
revision `bcb17bb25360ed68ce3bc084645db3e929799389`, blob
`5cb5070d0ea667fcb51b1fe972258d8761f8240d`. Copyright 2024 Google LLC,
Apache-2.0. All 48 source combinations are retained through FlagQuantum-owned
implementations.

Cycle counts, durations and footprints match the reference exactly. Full complex
density matrices use absolute tolerance 2e-15. At 400 million T-equivalent states,
the observed largest total-error difference is 3.9968028886505635e-7, below the
derived absolute 8e-7 bound. Source strict budget-rounding behavior is recorded
separately in `SURFACE_CODE_SCHEME_FILE_AUDIT.md`.
